# 🤖 Models

This notebook shows how to use the various network architectures defined in this project.

## Setup

---

Let's install some necessary dependencies and set global variables.

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Autoroot
import autorootcwd

In [3]:
# Imports
import torch

from src.models.net import FFNN, Conv, UNet, AutoTranslate

## Models


### FFNN

The `FFNN` is a simple linear encoder-decoder network.

In [4]:
# Initialise `FFNN`
net = FFNN(input_output_size=32*32, hidden_dims=[256, 128])
net

FFNN(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=1024, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
    )
    (1): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
    )
  )
  (decoder): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
    )
    (1): Sequential(
      (0): Linear(in_features=256, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
    )
  )
)

In [5]:
# Forward pass
x = torch.randn(1, 1, 32, 32)
y = net(x)

assert x.shape == y.shape
x.shape, y.shape

(torch.Size([1, 1, 32, 32]), torch.Size([1, 1, 32, 32]))

### Conv


A simple convolutional encoder-decoder neural network (without skip connections)

In [6]:
# Initialise `Conv`
net = Conv(input_output_channels=3, hidden_channels=[64, 128, 256])
net

Conv(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
  )
  (decoder): ModuleList(
    (0): ModuleList(
      (0): ConvTranspose2d(256, 128, kernel_size=(2, 2), stride=(2, 2))
      (1): Sequential(
        (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU()
        (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), p

In [7]:
# Forward pass
x = torch.randn(1, 3, 32, 32)
y = net(x)

assert x.shape == y.shape
x.shape, y.shape

(torch.Size([1, 3, 32, 32]), torch.Size([1, 3, 32, 32]))

### UNet

In [8]:
# Initalise UNet
net = UNet(input_output_channels=3, hidden_channels=[64, 128, 256, 512])
net

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
  )
  (bottleneck): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (decoder): ModuleList(
    (0): ModuleList(
      (0): ConvTranspose2d(512, 256, kernel_

In [9]:
# Forward pass
x = torch.randn(1, 3, 32, 32)
y = net(x)

assert x.shape == y.shape
x.shape, y.shape

(torch.Size([1, 3, 32, 32]), torch.Size([1, 3, 32, 32]))

### AutoTranslate

The AutoTranslateNet is an implementation of the network seen in
'Semi-Supervised Raw-to-Raw mapping' https://arxiv.org/pdf/2106.13883

The network consists of two auto-encoders, one for the digital domain and
one for the film domain.

There is then a translation network that is trained to map the latent space
of the source domain auto-encoder to the latent space of the target domain
auto-encoder.

In [10]:
# Initialise `AutoTranslate`
net = AutoTranslate(input_output_channels=3, hidden_channels=[64, 128, 256, 512])
net

AutoTranslate(
  (digital_autoencoder): UNet(
    (encoder): ModuleList(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU()
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU()
      )
      (1): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU()
        (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU()
      )
      (2): Sequential(
        (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU()
        (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU()
      )
    )
    (bottleneck): Sequential(
      (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    

In [11]:
# Forward pass
film = torch.randn(1, 3, 32, 32)
digital = torch.randn(1, 3, 32, 32)
paired = (torch.randn(1, 3, 32, 32), torch.randn(1, 3, 32, 32))

(
    digital_reconstructed,
    film_reconstructed,
    digital_to_film,
    film_to_digital,
    paired_encoder_representations,
) = net(digital, film, paired)